In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

First Steps - prompt User / prompt System

In [2]:


load_dotenv()
LLM_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

resp = LLM_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Tu es un assistant utile."},
        {"role": "user", "content": "Tu es un assistant clair et pédagogique. Réponds en anglais."},
    ],
)

print(resp.choices[0].message.content)

Of course! I'm here to help you. How can I assist you today?


In [3]:
resp = LLM_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Tu es un assistant utile, tu ne réponds qu'en anglais."},
        {"role": "user", "content": "Tu es un assistant clair et pédagogique."},
    ],
)

print(resp.choices[0].message.content)

I appreciate your feedback! How can I assist you today?


In [4]:
resp = LLM_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Tu es un assistant utile, tu ne réponds qu'en anglais."},
        {"role": "user", "content": "Tu es un assistant clair et pédagogique, réponds moi en francais."},
    ],
)

print(resp.choices[0].message.content)

I'm here to help you in English. How can I assist you today?


In [ ]:
def ask_llm(question: str) -> str:
    """
    Takes a question as input and returns an answer from the LLM.
    """
    response = LLM_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Tu es un assistant clair et pédagogique."
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=0.2,
    )
    
    answer = response.choices[0].message.content

    print(answer)

In [6]:
ask_llm("Bonjour je m'appelle Jerome")

Bonjour Jérôme ! Comment puis-je vous aider aujourd'hui ?


In [7]:
ask_llm("Comment je m'appelle ?")

Je ne connais pas votre nom, mais je serais ravi de le savoir si vous souhaitez le partager ! Comment puis-je vous aider aujourd'hui ?


Add memory

In [ ]:
def ask_llm_with_memory(
    question: str,
    memory: list | None = None
):
    """
    Send a question to the LLM with:
    - a system prompt that is ALWAYS present
    - a user/assistant memory
    """
    SYSTEM_PROMPT = "Tu es un assistant clair et pédagogique."
    
    if memory is None:
        memory = []

    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + list(memory)
        + [{"role": "user", "content": question}]
    )

    model_response = LLM_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.2,
    )

    answer = model_response.choices[0].message.content

    updated_memory = list(memory) + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]

    return model_response, answer, updated_memory

In [9]:
memory = []
model_response, answer, memory = ask_llm_with_memory("Bonjour, je m'appelle Jerome", memory)
print(answer)


Bonjour Jérôme ! Comment puis-je vous aider aujourd'hui ?


In [10]:
model_response, answer,memory = ask_llm_with_memory("Comment je m'appelle ?", memory)
print(answer)

Vous vous appelez Jérôme. Comment puis-je vous aider aujourd'hui ?


In [11]:
print(memory)

[{'role': 'user', 'content': "Bonjour, je m'appelle Jerome"}, {'role': 'assistant', 'content': "Bonjour Jérôme ! Comment puis-je vous aider aujourd'hui ?"}, {'role': 'user', 'content': "Comment je m'appelle ?"}, {'role': 'assistant', 'content': "Vous vous appelez Jérôme. Comment puis-je vous aider aujourd'hui ?"}]


INTERACT WITH PDF

1. Loading a training database : firefighter doctrine

In [ ]:
import requests
from pathlib import Path
from urllib.parse import urlparse

def download_pdf(url: str) -> str:
    """
    Downloads a PDF to ../data/pdf/ using the filename from the URL.
    Creates the folder if necessary.
    Returns the local path to the PDF.
    """
    filename = Path(urlparse(url).path).name
    output_dir = Path("../data/pdf")
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / filename

    response = requests.get(url)
    response.raise_for_status()
    output_path.write_bytes(response.content)

    return str(output_path)

In [13]:
import fitz
import re
from pathlib import Path

def pdf_to_text(pdf_path: str, max_pages: int | None = None) -> str:
    """
    Extrait le texte d'un PDF, nettoie les lignes vides successives,
    et l'enregistre dans ../data/raw_text/.
    Retourne le texte extrait.
    """
    pdf_path = Path(pdf_path)
    output_dir = Path("../data/raw_text")
    output_dir.mkdir(parents=True, exist_ok=True)

    output_txt_path = output_dir / f"{pdf_path.stem}.txt"

    doc = fitz.open(pdf_path)
    texts = []

    n = doc.page_count if max_pages is None else min(max_pages, doc.page_count)
    for i in range(n):
        page = doc.load_page(i)
        texts.append(page.get_text("text"))

    doc.close()

    # concat brut
    full_text = "\n\n".join(texts)

    # 🔧 nettoyage : lignes vides multiples → une seule
    full_text = re.sub(r"\n\s*\n+", "\n\n", full_text)

    output_txt_path.write_text(full_text, encoding="utf-8")

    return full_text

In [14]:
from typing import List

def download_and_extract_pdfs(urls: List[str], max_pages: int | None = None):
    """
    Télécharge une liste de PDFs et génère leur version texte.
    Les PDFs sont stockés dans ../data/pdf/
    Les textes sont stockés dans ../data/raw_text/
    """
    results = []

    for i, url in enumerate(urls, start=1):
        print(f"\n📄 [{i}/{len(urls)}] Traitement : {url}")

        try:
            pdf_path = download_pdf(url)
            print(f"   ⬇️ PDF téléchargé : {pdf_path}")

            text = pdf_to_text(pdf_path, max_pages=max_pages)
            print(f"   📝 Texte extrait ({len(text)} caractères)")

            results.append({
                "url": url,
                "pdf_path": pdf_path,
                "text_length": len(text),
                "status": "ok"
            })

        except Exception as e:
            print(f"   ❌ Erreur : {e}")
            results.append({
                "url": url,
                "status": "error",
                "error": str(e)
            })

    return results

In [15]:
pdf_urls = [
    "https://soldatdufeu.fr/wp-content/uploads/2024/01/GDO-Operations-Presence-Electricite.pdf",
    "https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC.pdf",
    "https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2.pdf",
    "https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO-interventions-silos-VF-09-2019.pdf",
    "https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO_Interventions_dans_les_eoliennes_2019.pdf",
    "https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO-Interventions-en-milieu-agricole-2019-V2.pdf",
    "https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO_interventions_a_bord_bateaux_en_eaux_interieures.pdf",
]


In [16]:
download_and_extract_pdfs(pdf_urls)


📄 [1/7] Traitement : https://soldatdufeu.fr/wp-content/uploads/2024/01/GDO-Operations-Presence-Electricite.pdf
   ⬇️ PDF téléchargé : ..\data\pdf\GDO-Operations-Presence-Electricite.pdf
   📝 Texte extrait (219195 caractères)

📄 [2/7] Traitement : https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC.pdf
   ⬇️ PDF téléchargé : ..\data\pdf\GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC.pdf
   📝 Texte extrait (74651 caractères)

📄 [3/7] Traitement : https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2.pdf
   ⬇️ PDF téléchargé : ..\data\pdf\GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2.pdf
   📝 Texte extrait (179526 caractères)

📄 [4/7] Traitement : https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO-interventions-silos-VF-09-2019.pdf
   ⬇️ PDF téléchargé : ..\data\pdf\GDO-interventions-silos-VF-09-2019.pdf
   📝 Texte extrait (85832 caractères)

📄 [5/7] Traitement 

[{'url': 'https://soldatdufeu.fr/wp-content/uploads/2024/01/GDO-Operations-Presence-Electricite.pdf',
  'pdf_path': '..\\data\\pdf\\GDO-Operations-Presence-Electricite.pdf',
  'text_length': 219195,
  'status': 'ok'},
 {'url': 'https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC.pdf',
  'pdf_path': '..\\data\\pdf\\GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC.pdf',
  'text_length': 74651,
  'status': 'ok'},
 {'url': 'https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2.pdf',
  'pdf_path': '..\\data\\pdf\\GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2.pdf',
  'text_length': 179526,
  'status': 'ok'},
 {'url': 'https://soldatdufeu.fr/wp-content/uploads/2023/11/GDO-interventions-silos-VF-09-2019.pdf',
  'pdf_path': '..\\data\\pdf\\GDO-interventions-silos-VF-09-2019.pdf',
  'text_length': 85832,
  'status': 'ok'},
 {'url': 'https://soldatdufeu.fr/wp-content/uploads/

2. Querying a predefined PDF with successive questions

In [ ]:
def ask_question_on_doc_naive(
    question: str,
    doc_stem: str = "GDO-Operations-Presence-Electricite",
    memory: list | None = None
):
    """
    Ask a question about a document in naive mode:
    - loads the full text from ../data/raw_text/{doc_stem}.txt
    - injects all the text into the question
    - uses ask_llm_with_memory
    - returns (answer, updated_memory)
    """
    
    txt_path = Path("../data/raw_text") / f"{doc_stem}.txt"
    if not txt_path.exists():
        raise FileNotFoundError(f"Fichier texte introuvable : {txt_path}")

    doc_text = txt_path.read_text(encoding="utf-8")

    full_question = f"""
Tu dois répondre uniquement à partir du document ci-dessous.
Si l'information n'est pas présente, réponds explicitement : "Je ne sais pas".

=== DOCUMENT : {doc_stem} ===
{doc_text}

=== QUESTION ===
{question}
""".strip()

    model_response, answer, updated_memory = ask_llm_with_memory(full_question, memory)

    return model_response, answer, updated_memory

In [18]:
memory = None

model_response, answer, memory = ask_question_on_doc_naive(
    "Quels sont les principaux risques liés à la présence d'électricité en intervention ? Réponse ultra synthétique",
    memory=memory
)

print("\n--- RÉPONSE ---")
print(answer)


--- RÉPONSE ---
Les principaux risques liés à la présence d'électricité en intervention sont :

1. **Choc électrique** : Contact direct ou indirect avec des conducteurs sous tension.
2. **Électrisation** : Passage de courant à travers le corps, pouvant entraîner des brûlures ou des arrêts cardiaques.
3. **Arc électrique** : Décharges pouvant provoquer des brûlures et des explosions.
4. **Incendies** : Départ de feu dû à des installations défectueuses ou à des arcs électriques.
5. **Tension de pas** : Risque d'électrisation en se tenant près d'un point d'entrée de courant dans le sol.
6. **Foudroiement** : Risques liés aux décharges électriques naturelles.


In [19]:
model_response, answer, memory = ask_question_on_doc_naive(
    "Dis m'en plus sur la tension de pas, en une ou deux phrases",
    memory=memory
)
print("\n--- RÉPONSE ---")
print(answer)


--- RÉPONSE ---
La tension de pas est la différence de tension entre les pieds d'une personne se tenant debout près d'un point d'entrée de courant à la terre, ce qui peut entraîner une électrisation par contact simultané des deux pieds avec le sol. Elle se manifeste lorsque le courant s'écoule dans la terre, se diffusant autour du point de contact, et représente un risque pour les personnes à proximité.


In [20]:
model_response, answer, memory = ask_question_on_doc_naive(
    "Comment j'évites ce risque ?",
    memory=memory
)
print("\n--- RÉPONSE ---")
print(answer)

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 128000 tokens. However, your messages resulted in 152277 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

3. Querying a corpus of documents

In [21]:
def ask_question_on_all_docs_naive(
    question: str,
    memory: list | None = None
):
    """
    Pose une question sur l'ensemble des documents en mode naïf :
    - charge tous les fichiers .txt depuis ../data/raw_text/
    - concatène tous les textes
    - injecte tout le corpus dans la question
    - utilise ask_llm_with_memory
    - retourne (answer, updated_memory)
    """
    raw_text_dir = Path("../data/raw_text")
    if not raw_text_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable : {raw_text_dir}")

    all_texts = []
    for txt_path in sorted(raw_text_dir.glob("*.txt")):
        doc_text = txt_path.read_text(encoding="utf-8")
        all_texts.append(
            f"\n\n===== DOCUMENT : {txt_path.stem} =====\n\n{doc_text}"
        )

    full_corpus = "\n".join(all_texts)

    full_question = f"""
Tu dois répondre uniquement à partir du corpus ci-dessous.
Si l'information n'est pas présente, réponds explicitement : "Je ne sais pas".

=== CORPUS COMPLET ===
{full_corpus}

=== QUESTION ===
{question}
""".strip()

    model_response, answer, memory = ask_llm_with_memory(full_question, memory)

    return model_response, answer, memory

In [22]:
memory = None

model_response, answer, memory = ask_question_on_all_docs_naive(
    "Quelles sont les règles générales de sécurité communes à toutes les interventions ?",
    memory=memory
)

print(answer)

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 128000 tokens. However, your messages resulted in 198760 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

3) Master contexte size

In [23]:
import tiktoken
def count_tokens_for_doc(doc_stem: str = "GDO-Operations-Presence-Electricite", encoding_name: str = "o200k_base") -> int:
    """Calcule le nombre de tokens d'un document texte stocké dans ../data/raw_text/."""
    txt_path = Path("../data/raw_text") / f"{doc_stem}.txt"

    text = txt_path.read_text(encoding="utf-8")

    enc = tiktoken.get_encoding(encoding_name)
    n_tokens = len(enc.encode(text))

    print(f"📄 {doc_stem} → {n_tokens:,} tokens")

In [24]:
count_tokens_for_doc("GDO-Operations-Presence-Electricite")

📄 GDO-Operations-Presence-Electricite → 50,605 tokens


In [25]:
memory = None

model_response, answer, memory = ask_question_on_doc_naive(
    "Quelle est la distance de sécurité à respecter lors d'une intervention à proximité d'une ligne électrique aérienne ? Réponse synthétique",
    memory=memory
)

print("\n--- RÉPONSE ---")
print(answer)

print("Prompt tokens     :", model_response.usage.prompt_tokens)
print("Completion tokens :", model_response.usage.completion_tokens)
print("Total tokens      :", model_response.usage.total_tokens)


--- RÉPONSE ---
La distance de sécurité à respecter lors d'une intervention à proximité d'une ligne électrique aérienne est de 10 mètres.
Prompt tokens     : 50698
Completion tokens : 24
Total tokens      : 50722
